# Cosine Similarity Edge Graph Analysis
## Does Script Representation Matter? — Experiment 1 Extension

**Three-language graph analysis pipeline**: Elamite, Akkadian, Sumerian

This notebook builds Word2Vec embeddings → cosine similarity graphs → bigram networks →
Louvain community detection, then compares graph structure across languages and
(optionally) across Latin vs Unicode representations.

**Outputs:**
- Per-language graph statistics (nodes, edges, modularity)
- Community ↔ POS alignment analysis
- Reinforced edge detection (fixed constructions)
- Cross-language comparison table
- Exportable CSV/JSON for each language

---

## 0. Setup & Configuration

In [4]:
# ═══════════════════════════════════════════════════════════════
# 0a. MOUNT GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = 'data/'

# Install dependencies (uncomment if needed)
!pip install gensim scikit-learn networkx matplotlib seaborn

import csv
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from gensim.models import Word2Vec
import networkx as nx
from networkx.algorithms import community as nx_community
from sklearn.metrics import adjusted_mutual_info_score
from IPython.display import display, HTML

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

print('All imports loaded.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.4 MB/s eta 0:00:00
All imports loaded.


## 1. Configuration

Set file paths and analysis parameters. Edit these paths to match your local setup.

In [5]:
# ═══════════════════════════════════════════════════════════════
# FILE PATHS — edit these to point to your data files
# ═══════════════════════════════════════════════════════════════

DATA_CONFIG = {
    'elamite': {
        'file': BASE_PATH + 'UnTN-Nasu texts Word-level.csv',
        'text_col': 'id_text',
        'form_col': 'Text',
        'pos_col': None,           # No POS in this CSV; set to column name if available
        'lang_filter': None,
        'max_tokens': None,        # Use all (small corpus)
    },
    'akkadian': {
        'file': BASE_PATH + 'alltexts_AKK.csv',
        'text_col': 'id_text',
        'form_col': 'form',
        'pos_col': 'pos',
        'lang_filter': None,
        'max_tokens': None,      # Subsample for tractability
    },
    'sumerian': {
        'file': BASE_PATH + 'alltexts_SUX -alltexts.csv',
        'text_col': 'id_text',
        'form_col': 'form',
        'pos_col': 'pos',
        'lang_filter': None,
        'max_tokens': None,
    },
}

# ═══════════════════════════════════════════════════════════════
# ANALYSIS PARAMETERS
# ═══════════════════════════════════════════════════════════════

W2V_PARAMS = {
    'vector_size': 100,
    'window': 5,
    'min_count': 3,      # Minimum word frequency for W2V vocab
    'workers': 4,
    'epochs': 20,
    'sg': 1,             # Skip-gram
}

BIGRAM_WEIGHT = 0.15             # Weight per bigram occurrence
SIMILARITY_THRESHOLD = 0.40     # Min cosine similarity for edge
SIMILARITY_MULTIPLIER = 1.5     # Scale factor for similarity weights
MIN_TEXT_LENGTH = 3              # Min words per text to include
SKIP_POS = {'u', 'X', ''}       # POS tags to skip (unlemmatized, unknown)

OUTPUT_DIR = BASE_PATH + 'graph_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuration set.')
print(f'Output directory: {OUTPUT_DIR}/')

Configuration set.
Output directory: data/graph_outputs/


## 2. Data Loading

Unified loader that handles all three CSV formats.

In [7]:
def load_corpus(config, language_name):
    """
    Load a cuneiform corpus CSV into document sequences.

    Returns:
        documents: dict {text_id: [word1, word2, ...]}
        word_pos:  dict {word: POS_tag}
    """
    filepath = config['file']
    text_col = config['text_col']
    form_col = config['form_col']
    pos_col = config['pos_col']
    max_tokens = config.get('max_tokens')

    if not os.path.exists(filepath):
        print(f'  ⚠ File not found: {filepath} — skipping {language_name}')
        return None, None

    documents = defaultdict(list)
    word_pos = {}
    total_tokens = 0

    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            form = row.get(form_col, '').strip()
            text_id = row.get(text_col, '').strip()
            pos = row.get(pos_col, '') if pos_col else ''
            pos = pos.strip() if pos else ''

            if not form or not text_id:
                continue
            if pos_col and pos in SKIP_POS:
                continue

            documents[text_id].append(form)
            if pos and form not in word_pos:
                word_pos[form] = pos

            total_tokens += 1
            if max_tokens and total_tokens >= max_tokens:
                break

    # Filter short texts
    documents = {k: v for k, v in documents.items() if len(v) >= MIN_TEXT_LENGTH}

    n_tokens = sum(len(w) for w in documents.values())
    n_unique = len(set(w for ws in documents.values() for w in ws))

    print(f'  Loaded {language_name}: {len(documents)} texts, '
          f'{n_tokens:,} tokens, {n_unique:,} unique words')
    if word_pos:
        pos_dist = Counter(word_pos.values())
        print(f'  POS distribution: {dict(pos_dist.most_common(8))}')

    return dict(documents), word_pos

# ── Load all corpora ──
corpora = {}
for lang, config in DATA_CONFIG.items():
    print(f'\nLoading {lang}...')
    docs, wpos = load_corpus(config, lang)
    if docs is not None:
        corpora[lang] = {'documents': docs, 'word_pos': wpos}

print(f'\n✓ Loaded {len(corpora)} corpora: {", ".join(corpora.keys())}')


Loading elamite...
  Loaded elamite: 0 texts, 0 tokens, 0 unique words

Loading akkadian...
  Loaded akkadian: 13092 texts, 1,232,552 tokens, 117,096 unique words
  POS distribution: {'N': 40998, 'V': 28155, 'PN': 19465, 'AJ': 7622, 'n': 2863, 'SN': 2783, 'GN': 2354, 'DN': 2275}

Loading sumerian...
  ⚠ File not found: data/alltexts_SUX -alltexts.csv — skipping sumerian

✓ Loaded 2 corpora: elamite, akkadian


## 3. Graph Analysis Pipeline

Core function that runs the full analysis for one language:
1. Train Word2Vec
2. Build bigram edges (syntagmatic)
3. Build similarity edges (paradigmatic)
4. Combine into unified graph
5. Louvain community detection
6. Centrality metrics
7. Community ↔ POS alignment

In [8]:
def run_graph_analysis(documents, word_pos, language):
    """
    Full graph analysis pipeline for one language.

    Returns a results dict with all metrics and the NetworkX graph.
    """
    print(f'\n{"=" * 60}')
    print(f' {language.upper()} GRAPH ANALYSIS')
    print(f'{"=" * 60}')

    sentences = list(documents.values())
    all_tokens = sum(len(s) for s in sentences)
    unique_words = set(w for s in sentences for w in s)

    # ── 1. Train Word2Vec ──
    print(f'\n[1] Training Word2Vec ({all_tokens:,} tokens)...')
    w2v = Word2Vec(sentences, **W2V_PARAMS)
    vocab = set(w2v.wv.index_to_key)
    print(f'    Vocabulary: {len(vocab)} words')

    # ── 2. Build bigram edges ──
    print(f'[2] Building bigram edges...')
    bigram_counts = Counter()
    for words in sentences:
        for i in range(len(words) - 1):
            if words[i] in vocab and words[i+1] in vocab:
                bigram_counts[tuple(sorted([words[i], words[i+1]]))] += 1
    print(f'    Unique bigram pairs: {len(bigram_counts):,}')

    # ── 3. Build similarity edges ──
    print(f'[3] Building similarity edges (threshold={SIMILARITY_THRESHOLD})...')
    sim_edges = {}
    for w1 in vocab:
        try:
            for w2, sim in w2v.wv.most_similar(w1, topn=10):
                if sim >= SIMILARITY_THRESHOLD:
                    edge = tuple(sorted([w1, w2]))
                    if edge not in sim_edges or sim > sim_edges[edge]:
                        sim_edges[edge] = sim
        except KeyError:
            continue
    print(f'    Similarity edges: {len(sim_edges):,}')

    # ── 4. Build combined graph ──
    print(f'[4] Building combined graph...')
    G = nx.Graph()
    for (w1, w2), count in bigram_counts.items():
        G.add_edge(w1, w2, bigram=count * BIGRAM_WEIGHT, similarity=0.0)
    for (w1, w2), sim in sim_edges.items():
        if G.has_edge(w1, w2):
            G[w1][w2]['similarity'] = sim * SIMILARITY_MULTIPLIER
        else:
            G.add_edge(w1, w2, bigram=0.0, similarity=sim * SIMILARITY_MULTIPLIER)
    for u, v, d in G.edges(data=True):
        d['weight'] = d['bigram'] + d['similarity']

    n_both = sum(1 for _, _, d in G.edges(data=True) if d['bigram'] > 0 and d['similarity'] > 0)
    n_bigram = sum(1 for _, _, d in G.edges(data=True) if d['bigram'] > 0 and d['similarity'] == 0)
    n_sim = sum(1 for _, _, d in G.edges(data=True) if d['bigram'] == 0 and d['similarity'] > 0)
    print(f'    Nodes: {G.number_of_nodes():,}, Edges: {G.number_of_edges():,}')
    print(f'    Bigram-only: {n_bigram:,}, Similarity-only: {n_sim:,}, Reinforced: {n_both:,}')

    # ── 5. Community detection ──
    print(f'[5] Louvain community detection...')
    communities = nx_community.louvain_communities(G, weight='weight', seed=42)
    node_community = {}
    for i, comm in enumerate(communities):
        for node in comm:
            node_community[node] = i
    modularity = nx_community.modularity(G, communities, weight='weight')
    comm_sizes = sorted([len(c) for c in communities], reverse=True)
    print(f'    Communities: {len(communities)}, Modularity: {modularity:.4f}')
    print(f'    Sizes (top 10): {comm_sizes[:10]}')

    # ── 6. Centrality ──
    print(f'[6] Computing centrality metrics...')
    eigenvector = nx.eigenvector_centrality_numpy(G, weight='weight')
    k_betw = min(300, G.number_of_nodes())
    betweenness = nx.betweenness_centrality(G, weight='weight', k=k_betw)

    # ── 7. Community ↔ POS alignment ──
    ami_score = None
    if word_pos:
        print(f'[7] Community ↔ POS alignment...')
        words_with_both = [w for w in G.nodes() if w in word_pos and w in node_community]
        if len(words_with_both) > 50:
            true_labels = [word_pos[w] for w in words_with_both]
            comm_labels = [node_community[w] for w in words_with_both]
            ami_score = adjusted_mutual_info_score(true_labels, comm_labels)
            print(f'    AMI (Community vs POS): {ami_score:.4f}')

        print(f'\n    Community breakdown:')
        for cid in range(min(10, len(communities))):
            members = [n for n, c in node_community.items() if c == cid]
            pos_dist = Counter(word_pos.get(m, '?') for m in members)
            top = sorted(members, key=lambda n: eigenvector.get(n, 0), reverse=True)[:5]
            print(f'    C{cid:2d} ({len(members):4d} words): '
                  f'{dict(pos_dist.most_common(4))}')
            print(f'         Top: {", ".join(top)}')

    # ── Reinforced edges ──
    reinforced = [(u, v, d) for u, v, d in G.edges(data=True)
                  if d['bigram'] > 0 and d['similarity'] > 0]
    reinforced.sort(key=lambda x: x[2]['weight'], reverse=True)

    print(f'\n    Top 10 reinforced edges (fixed constructions):')
    for u, v, d in reinforced[:10]:
        same = 'SAME' if node_community.get(u) == node_community.get(v) else 'DIFF'
        pu, pv = word_pos.get(u, '?'), word_pos.get(v, '?')
        print(f'    {u:22s}({pu:3s}) — {v:22s}({pv:3s}) '
              f'w={d["weight"]:.2f} [{same}]')

    # ── Package results ──
    results = {
        'language': language,
        'graph': G,
        'w2v': w2v,
        'communities': communities,
        'node_community': node_community,
        'eigenvector': eigenvector,
        'betweenness': betweenness,
        'word_pos': word_pos,
        'stats': {
            'texts': len(documents),
            'tokens': all_tokens,
            'unique_words': len(unique_words),
            'vocab': len(vocab),
            'nodes': G.number_of_nodes(),
            'edges': G.number_of_edges(),
            'bigram_only': n_bigram,
            'similarity_only': n_sim,
            'reinforced': n_both,
            'n_communities': len(communities),
            'modularity': round(modularity, 4),
            'ami_vs_pos': round(ami_score, 4) if ami_score is not None else None,
            'community_sizes': comm_sizes,
        },
        'reinforced_edges': [(u, v, d) for u, v, d in reinforced],
    }

    print(f'\n✓ {language} analysis complete.')
    return results

## 4. Run Analysis on All Languages

In [9]:
# Run the pipeline on each loaded corpus
results = {}

for lang, corpus in corpora.items():
    results[lang] = run_graph_analysis(
        corpus['documents'],
        corpus['word_pos'],
        lang
    )

print(f'\n{"=" * 60}')
print(f'All {len(results)} languages processed.')
print(f'{"=" * 60}')


 ELAMITE GRAPH ANALYSIS

[1] Training Word2Vec (0 tokens)...


RuntimeError: you must first build vocabulary before training the model

## 5. Cross-Language Comparison

Summary table comparing graph properties across all three languages.

In [10]:
# ── Build comparison table ──
comparison_rows = []
metrics = [
    ('Texts', 'texts'), ('Tokens', 'tokens'), ('Unique words', 'unique_words'),
    ('W2V vocab', 'vocab'), ('Graph nodes', 'nodes'), ('Graph edges', 'edges'),
    ('Bigram-only edges', 'bigram_only'), ('Similarity-only edges', 'similarity_only'),
    ('Reinforced edges', 'reinforced'), ('Communities', 'n_communities'),
    ('Modularity', 'modularity'), ('AMI (Community vs POS)', 'ami_vs_pos'),
]

print(f'{"Metric":<28s}', end='')
for lang in results:
    print(f'{lang:>15s}', end='')
print()
print('─' * (28 + 15 * len(results)))

for label, key in metrics:
    print(f'{label:<28s}', end='')
    for lang in results:
        val = results[lang]['stats'].get(key)
        if val is None:
            print(f'{"N/A":>15s}', end='')
        elif isinstance(val, float):
            print(f'{val:>15.4f}', end='')
        else:
            print(f'{val:>15,}', end='')
    print()

print('─' * (28 + 15 * len(results)))

Metric                      
────────────────────────────
Texts                       
Tokens                      
Unique words                
W2V vocab                   
Graph nodes                 
Graph edges                 
Bigram-only edges           
Similarity-only edges       
Reinforced edges            
Communities                 
Modularity                  
AMI (Community vs POS)      
────────────────────────────


## 6. Visualizations

### 6a. Modularity & Community Count Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

langs = list(results.keys())
colors = ['#534AB7', '#0F6E56', '#D85A30']  # purple, teal, coral

# ── Modularity ──
ax = axes[0]
mods = [results[l]['stats']['modularity'] for l in langs]
ax.bar(langs, mods, color=colors[:len(langs)], width=0.5)
ax.set_ylabel('Modularity')
ax.set_title('Graph Modularity')
ax.set_ylim(0, 1)
for i, v in enumerate(mods):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# ── Edge type distribution ──
ax = axes[1]
x = np.arange(len(langs))
w = 0.25
b_vals = [results[l]['stats']['bigram_only'] for l in langs]
s_vals = [results[l]['stats']['similarity_only'] for l in langs]
r_vals = [results[l]['stats']['reinforced'] for l in langs]
ax.bar(x - w, b_vals, w, label='Bigram only', color='#85B7EB')
ax.bar(x, s_vals, w, label='Similarity only', color='#F09595')
ax.bar(x + w, r_vals, w, label='Reinforced', color='#97C459')
ax.set_xticks(x)
ax.set_xticklabels(langs)
ax.set_ylabel('Edge count')
ax.set_title('Edge Type Distribution')
ax.legend(fontsize=9)

# ── Community size distributions ──
ax = axes[2]
for i, lang in enumerate(langs):
    sizes = results[lang]['stats']['community_sizes'][:15]
    ax.plot(range(len(sizes)), sizes, 'o-', color=colors[i], label=lang, markersize=4)
ax.set_xlabel('Community rank')
ax.set_ylabel('Community size')
ax.set_title('Community Size Distribution')
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR}/comparison_overview.png')

### 6b. Top Eigenvector Centrality Words (per language)

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 8))
if len(results) == 1:
    axes = [axes]

for ax, (lang, res) in zip(axes, results.items()):
    eig = res['eigenvector']
    wpos = res['word_pos']
    top20 = sorted(eig.items(), key=lambda x: -x[1])[:20]

    words = [w for w, _ in top20]
    scores = [s for _, s in top20]
    pos_labels = [wpos.get(w, '?') for w in words]

    # Color by POS
    pos_colors = {'N': '#85B7EB', 'V': '#F09595', 'AJ': '#97C459',
                  'PN': '#F5C4B3', 'DN': '#CECBF6', 'GN': '#9FE1CB',
                  'n': '#FAC775', 'CN': '#ED93B1'}
    bar_colors = [pos_colors.get(p, '#D3D1C7') for p in pos_labels]

    y_pos = range(len(words))
    ax.barh(y_pos, scores, color=bar_colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'{w} [{p}]' for w, p in zip(words, pos_labels)], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Eigenvector centrality')
    ax.set_title(f'{lang.title()} — Top 20 words')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eigenvector_centrality.png', dpi=150, bbox_inches='tight')
plt.show()

### 6c. Community ↔ POS Heatmap

In [ ]:
for lang, res in results.items():
    if not res['word_pos']:
        print(f'{lang}: No POS data, skipping heatmap')
        continue

    nc = res['node_community']
    wp = res['word_pos']

    # Build community × POS matrix
    all_pos = sorted(set(wp.values()))
    n_comms = min(15, res['stats']['n_communities'])

    matrix = np.zeros((n_comms, len(all_pos)))
    for word, comm in nc.items():
        if comm < n_comms and word in wp:
            pos_idx = all_pos.index(wp[word])
            matrix[comm, pos_idx] += 1

    # Normalize rows to proportions
    row_sums = matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    matrix_norm = matrix / row_sums

    fig, ax = plt.subplots(figsize=(max(10, len(all_pos) * 0.8), n_comms * 0.5 + 2))
    sns.heatmap(matrix_norm, xticklabels=all_pos, yticklabels=[f'C{i}' for i in range(n_comms)],
                cmap='YlOrRd', ax=ax, annot=matrix.astype(int), fmt='d',
                linewidths=0.5, cbar_kws={'label': 'Proportion'})
    ax.set_title(f'{lang.title()} — Community × POS Distribution')
    ax.set_xlabel('POS Tag')
    ax.set_ylabel('Community')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{lang}_community_pos_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

### 6d. Reinforced Edges Network (per language)

In [ ]:
for lang, res in results.items():
    reinforced = res['reinforced_edges'][:30]  # Top 30
    if not reinforced:
        continue

    # Build subgraph of reinforced edges
    H = nx.Graph()
    for u, v, d in reinforced:
        H.add_edge(u, v, weight=d['weight'])

    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.spring_layout(H, k=2.0, seed=42, weight='weight')

    # Node colors by community
    comm_colors = plt.cm.Set3(np.linspace(0, 1, 12))
    node_colors = [comm_colors[res['node_community'].get(n, 0) % 12] for n in H.nodes()]

    # Edge widths by weight
    weights = [H[u][v]['weight'] for u, v in H.edges()]
    max_w = max(weights) if weights else 1
    edge_widths = [1 + 3 * w / max_w for w in weights]

    nx.draw_networkx_edges(H, pos, width=edge_widths, alpha=0.5, edge_color='#888', ax=ax)
    nx.draw_networkx_nodes(H, pos, node_color=node_colors, node_size=300,
                           edgecolors='#333', linewidths=0.5, ax=ax)
    nx.draw_networkx_labels(H, pos, font_size=7, ax=ax)

    ax.set_title(f'{lang.title()} — Top 30 Reinforced Edges (Bigram + Similarity)',
                fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{lang}_reinforced_network.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Cross-Representation Edge Agreement (Latin vs Unicode)

*This section compares the similarity graph structure between Latin transliteration
and Unicode cuneiform representations of the same corpus. Run this after you have
both representations preprocessed.*

The key question: **Do the two scripts capture the same distributional relationships?**

In [ ]:
def compare_edge_agreement(results_latin, results_unicode, language, top_k=500):
    """
    Compare top-k similarity edges between Latin and Unicode graphs.

    Requires a word-level mapping between representations.
    Returns: Jaccard similarity, overlap count, and edge-level details.
    """
    G_lat = results_latin['graph']
    G_uni = results_unicode['graph']

    # Get top-k edges by similarity weight
    def top_sim_edges(G, k):
        sim_edges = [(u, v, d['similarity']) for u, v, d in G.edges(data=True)
                     if d['similarity'] > 0]
        sim_edges.sort(key=lambda x: -x[2])
        return set((min(u,v), max(u,v)) for u, v, _ in sim_edges[:k])

    lat_top = top_sim_edges(G_lat, top_k)
    uni_top = top_sim_edges(G_uni, top_k)

    overlap = lat_top & uni_top
    jaccard = len(overlap) / len(lat_top | uni_top) if lat_top | uni_top else 0

    print(f'\n{language} — Cross-Representation Edge Agreement')
    print(f'  Top-{top_k} Latin edges:   {len(lat_top)}')
    print(f'  Top-{top_k} Unicode edges: {len(uni_top)}')
    print(f'  Overlap:                {len(overlap)}')
    print(f'  Jaccard similarity:     {jaccard:.4f}')

    # Community alignment between representations
    lat_mod = results_latin['stats']['modularity']
    uni_mod = results_unicode['stats']['modularity']
    print(f'  Modularity (Latin):     {lat_mod:.4f}')
    print(f'  Modularity (Unicode):   {uni_mod:.4f}')

    return {
        'jaccard': jaccard,
        'overlap': len(overlap),
        'lat_edges': len(lat_top),
        'uni_edges': len(uni_top),
        'modularity_latin': lat_mod,
        'modularity_unicode': uni_mod,
    }

# ── Usage (uncomment when you have both representations): ──
# agreement = compare_edge_agreement(
#     results_latin=results['elamite_latin'],
#     results_unicode=results['elamite_unicode'],
#     language='Elamite'
# )

print('Cross-representation comparison function ready.')
print('Run compare_edge_agreement() after loading Latin and Unicode versions.')

## 8. Export Results

In [ ]:
for lang, res in results.items():
    stats = res['stats']
    eig = res['eigenvector']
    bet = res['betweenness']
    nc = res['node_community']
    wp = res['word_pos']
    G = res['graph']

    # ── Export nodes CSV ──
    node_file = f'{OUTPUT_DIR}/{lang}_nodes.csv'
    with open(node_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['word', 'pos', 'community', 'eigenvector', 'betweenness',
                         'degree', 'strength'])
        for node in sorted(G.nodes()):
            strength = sum(d['weight'] for _, _, d in G.edges(node, data=True))
            writer.writerow([
                node, wp.get(node, '?'), nc.get(node, -1),
                round(eig.get(node, 0), 6), round(bet.get(node, 0), 6),
                G.degree(node), round(strength, 4)
            ])

    # ── Export edges CSV ──
    edge_file = f'{OUTPUT_DIR}/{lang}_edges.csv'
    with open(edge_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['source', 'target', 'bigram_weight', 'similarity_weight',
                         'total_weight', 'edge_type'])
        for u, v, d in sorted(G.edges(data=True), key=lambda x: -x[2]['weight']):
            etype = ('reinforced' if d['bigram'] > 0 and d['similarity'] > 0
                     else 'bigram' if d['bigram'] > 0 else 'similarity')
            writer.writerow([u, v, round(d['bigram'], 4), round(d['similarity'], 4),
                            round(d['weight'], 4), etype])

    # ── Export summary JSON ──
    summary = {
        'language': lang,
        'stats': stats,
        'top_eigenvector': [(w, round(s, 6), wp.get(w, '?'))
                           for w, s in sorted(eig.items(), key=lambda x: -x[1])[:25]],
        'top_betweenness': [(w, round(s, 6), wp.get(w, '?'))
                           for w, s in sorted(bet.items(), key=lambda x: -x[1])[:25]],
        'reinforced_edges': [
            (u, v, round(d['weight'], 3), wp.get(u, '?'), wp.get(v, '?'))
            for u, v, d in res['reinforced_edges'][:30]
        ],
        'community_pos': {
            str(i): dict(Counter(wp.get(m, '?')
                         for m in [n for n, c in nc.items() if c == i]).most_common(5))
            for i in range(min(15, stats['n_communities']))
        }
    }
    json_file = f'{OUTPUT_DIR}/{lang}_graph_analysis.json'
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    # ── Save W2V model ──
    model_file = f'{OUTPUT_DIR}/{lang}_word2vec.model'
    res['w2v'].save(model_file)

    print(f'{lang}: {node_file}, {edge_file}, {json_file}, {model_file}')

print(f'\n✓ All exports saved to {OUTPUT_DIR}/')

## 9. Summary & Cross-Language Comparison Plot

In [ ]:
# ── Final comparison bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
langs = list(results.keys())
colors = ['#534AB7', '#0F6E56', '#D85A30']

# Reinforced edge ratio
ax = axes[0]
ratios = [results[l]['stats']['reinforced'] / max(results[l]['stats']['edges'], 1) * 100
          for l in langs]
ax.bar(langs, ratios, color=colors[:len(langs)], width=0.5)
ax.set_ylabel('Reinforced edges (%)')
ax.set_title('Reinforced Edge Ratio')
for i, v in enumerate(ratios):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# AMI scores (if available)
ax = axes[1]
amis = [results[l]['stats'].get('ami_vs_pos') for l in langs]
valid_langs = [l for l, a in zip(langs, amis) if a is not None]
valid_amis = [a for a in amis if a is not None]
if valid_amis:
    ax.bar(valid_langs, valid_amis, color=[colors[langs.index(l)] for l in valid_langs], width=0.5)
    ax.set_ylabel('Adjusted Mutual Information')
    ax.set_title('Community ↔ POS Alignment (AMI)')
    ax.set_ylim(0, max(valid_amis) * 1.3)
    for i, v in enumerate(valid_amis):
        ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No POS data available', transform=ax.transAxes, ha='center')
    ax.set_title('Community ↔ POS Alignment (AMI)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cross_language_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '═' * 60)
print(' ANALYSIS COMPLETE')
print('═' * 60)
print(f'\nAll outputs saved to: {OUTPUT_DIR}/')
print(f'Languages analyzed: {", ".join(results.keys())}')
print(f'\nNext steps:')
print(f'  1. Run Unicode conversion on each corpus')
print(f'  2. Re-run this pipeline on Unicode representations')
print(f'  3. Use compare_edge_agreement() in Section 7')
print(f'  4. Compare modularity, AMI, and edge overlap across scripts')